# Credit Policy Lab

**Question:** which loans should a lender approve, and what does that do to profit? I use historical LendingClub loans to build a default model, turn it into an approval policy that maximises profit, and check how robust that policy is to out-of-time drift and to stress in funding cost and credit losses. Put the raw data file in `credit_policy_lab/data/` first (see the README); the notebook then runs top to bottom locally or on Google Colab.

## Setup

Flags first, then imports. Every assumption that changes a result lives in one config dict, so a hash of it tells me when saved models are stale. Features come from an explicit allowlist of fields known at origination: anything not on the list cannot reach a model, and a check runs before every fit.

In [ ]:
DEMO_MODE = False      # True: sample 150k loans and use a small tree budget, for fast smoke runs
FORCE_RETRAIN = False  # True: ignore saved models and retrain them
SEED = 42              # one seed for every random step

import sys, os, json, time, hashlib, importlib, subprocess, warnings
from pathlib import Path


def ensure(pkg, module):
    """Import a module, pip-installing its package first if it is missing."""
    try:
        return importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return importlib.import_module(module)


REQUIRED = {"pandas": "pandas", "numpy": "numpy", "scikit-learn": "sklearn", "lightgbm": "lightgbm",
            "duckdb": "duckdb", "matplotlib": "matplotlib", "pyarrow": "pyarrow", "scipy": "scipy"}
for pkg, module in REQUIRED.items():
    ensure(pkg, module)

import numpy as np
import pandas as pd
import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/credit_policy_lab")
else:
    BASE = Path("credit_policy_lab")
DATA_DIR, CACHE_DIR, MODEL_DIR, REPORT_DIR = BASE / "data", BASE / "cache", BASE / "models", BASE / "reports"
FIG_DIR = REPORT_DIR / "figures"
for d in (DATA_DIR, CACHE_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"environment: {'Colab' if IN_COLAB else 'local'} | base folder: {BASE}")

# Keys are the names this notebook uses; values are the column names in the raw file.
# To drop in another loan dataset, edit the values. The economics layer needs the cash-flow columns.
SCHEMA = {
    # identifiers and dates
    "id": "id", "issue_d": "issue_d", "earliest_cr_line": "earliest_cr_line",
    # outcome and realised cash flows: target and P&L only, never features
    "loan_status": "loan_status", "funded_amnt": "funded_amnt", "total_pymnt": "total_pymnt",
    "total_rec_prncp": "total_rec_prncp", "total_rec_int": "total_rec_int",
    "total_rec_late_fee": "total_rec_late_fee", "recoveries": "recoveries",
    "collection_recovery_fee": "collection_recovery_fee",
    # contract terms set by the incumbent lender: economics and baseline only, never features
    "term": "term", "int_rate": "int_rate", "installment": "installment", "grade": "grade",
    # applicant fields known at origination: the only source of model features
    "loan_amnt": "loan_amnt", "emp_length": "emp_length", "home_ownership": "home_ownership",
    "annual_inc": "annual_inc", "verification_status": "verification_status", "purpose": "purpose",
    "dti": "dti", "delinq_2yrs": "delinq_2yrs", "inq_last_6mths": "inq_last_6mths", "open_acc": "open_acc",
    "pub_rec": "pub_rec", "revol_bal": "revol_bal", "revol_util": "revol_util", "total_acc": "total_acc",
    "fico_range_low": "fico_range_low", "fico_range_high": "fico_range_high",
}
TEXT_COLS = ["id", "issue_d", "earliest_cr_line"]  # read as text; dates are parsed after loading
CATEGORY_COLS = ["loan_status", "term", "grade", "emp_length", "home_ownership",
                 "verification_status", "purpose"]  # few distinct values, stored as categories to save memory

In [ ]:
CFG = {
    "load": {
        "demo_rows": 150_000,          # 36-month loans sampled when DEMO_MODE is on
    },
    "cohort": {
        "term_months": 36,             # one fixed horizon so every loan had the same time to default
        "start": "2012-01-01",         # earlier vintages used different credit policies and extra status labels
        "train_end": "2013-12-01",     # train: issue months from start up to here
        "valid_end": "2014-12-01",     # validation: after train_end up to here
        "test_end": "2015-12-01",      # test: after valid_end up to here
        "max_unresolved_share": 0.01,  # maturity filter: keep issue months with at most 1% unresolved loans
    },
    "features": {
        "caps": {"delinq_2yrs": 5, "inq_last_6mths": 6, "pub_rec": 3},  # thin right tails, capped
        "revol_util_clip": [0.0, 150.0],  # utilisation above 150% is almost always a data artefact
        "dti_clip": [0.0, 100.0],         # guards against sentinel values such as -1
        "loan_to_income_cap": 2.0,        # loans above twice annual income are rare and noisy
        "dti_peer_window": 1000,          # number of earlier same-purpose loans in the trailing DTI average
    },
    "scorecard": {
        "max_bins": 10,                # quantile bins per numeric feature before merging
        "min_share": 0.05,             # every bin holds at least 5% of train
        "woe_smoothing": 0.5,          # added to good and bad counts so a tiny bin never gives infinite WoE
        "iv_min": 0.02,                # below this a feature carries almost no signal
        "iv_max": 0.5,                 # above this a feature is suspiciously strong (possible leakage)
        "max_woe_corr": 0.7,           # drop the weaker feature of any pair more correlated than this
        "C_grid": [0.001, 0.01, 0.1, 1.0, 10.0],  # inverse L2 strength, chosen on validation AUC
        "base_score": 600,             # score that corresponds to the base good:bad odds
        "pdo": 40,                     # points to double the odds
    },
}

In [ ]:
CFG.update({
    "lgbm": {
        "params": {"objective": "binary", "metric": "auc", "learning_rate": 0.05, "num_leaves": 31,
                   "min_child_samples": 200, "feature_fraction": 0.8, "bagging_fraction": 0.8,
                   "bagging_freq": 1, "lambda_l2": 10.0, "seed": SEED, "deterministic": True,
                   "force_row_wise": True, "verbose": -1},  # fixed, sensible values; only the tree count is tuned
        "num_rounds": 3000,            # upper bound; early stopping picks the actual number
        "demo_rounds": 150,            # small tree budget for DEMO_MODE
        "early_stopping": 100,         # stop after this many rounds without validation AUC improvement
        "monotone": {"fico_avg": -1, "log_income": -1, "dti": 1, "revol_util": 1, "inq_last_6mths": 1,
                     "delinq_2yrs": 1, "pub_rec": 1, "loan_to_income": 1},  # only where the sign is unambiguous
    },
    "eval": {
        "n_bootstrap": 200,            # resamples for the test AUC intervals
        "leakage_auc": 0.80,           # an origination-only model above this almost certainly leaks
        "psi_bins": 10,                # quantile bins for PSI
        "calib_tolerance": 0.10,       # mean PD within 10% of the actual bad rate counts as calibrated in the memo
    },
    "economics": {
        "cof": 0.05,                   # annual cost of funds
        "avg_life_eq": 1.65,           # years of average balance outstanding: 3 years x 0.55 balance factor
        "opex_pct": 0.015,             # servicing and origination cost as a share of funded amount
        "n_risk_bands": 6,             # bands for the P&L-by-risk table
        "hurdle_quantiles": 101,       # candidate EV hurdles are this many quantiles of validation EV
        "explain_margin": 0.05,        # scorecard within 5% of LightGBM's validation profit counts as close
    },
    "stress": {
        "cof_add": [0.0, 0.02, 0.04],  # funding cost shocks added to the base rate
        "pd_mult": [1.0, 1.25, 1.5],   # multipliers on every calibrated PD
        "bad_mult": [1.0, 1.25, 1.5],  # multipliers on the loss per dollar of a bad loan
        "breakeven_max": 10.0,         # search range for the breakeven PD multiplier
    },
})

In [ ]:
RAW_ALLOWLIST = ["loan_amnt", "emp_length", "home_ownership", "annual_inc", "verification_status", "purpose",
                 "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util",
                 "total_acc", "fico_range_low", "fico_range_high", "earliest_cr_line"]
DERIVED = ["fico_avg", "credit_history_years", "log_income", "loan_to_income", "log_revol_bal", "open_acc_ratio",
           "emp_length_years", "emp_length_missing", "inc_pctile_peer", "dti_vs_trailing_peer"]
ALLOWLIST = set(RAW_ALLOWLIST) | set(DERIVED)

# the incumbent's price and everything observed after origination
BANNED = ["int_rate", "grade", "sub_grade", "installment", "loan_status", "bad", "funded_amnt", "total_pymnt",
          "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee"]

NUM_FEATURES = ["loan_amnt", "log_income", "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
                "log_revol_bal", "revol_util", "total_acc", "fico_avg", "credit_history_years", "loan_to_income",
                "open_acc_ratio", "emp_length_years", "emp_length_missing", "inc_pctile_peer",
                "dti_vs_trailing_peer"]
CAT_FEATURES = ["home_ownership", "verification_status", "purpose"]
MODEL_FEATURES = NUM_FEATURES + CAT_FEATURES
ECON_COLS = ["loan_status", "funded_amnt", "total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
             "recoveries", "collection_recovery_fee", "term_months", "int_rate", "installment", "grade"]
SPLITS = ["train", "valid", "test"]


def assert_features_ok(cols):
    """Stop if any column outside the allowlist, or any banned column, is about to reach a model."""
    cols = list(cols)
    outside = sorted(set(cols) - ALLOWLIST)
    banned = sorted(set(cols) & set(BANNED))
    assert not outside, f"columns outside the allowlist: {outside}"
    assert not banned, f"banned columns: {banned}"


assert_features_ok(MODEL_FEATURES)

In [ ]:
# one colour per model in every chart, so a line means the same thing everywhere
COLORS = {"scorecard": "#1f77b4", "lgbm": "#d62728", "baseline": "#7f7f7f", "ev": "#2ca02c", "bar": "#9fb8d6"}
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.titlesize": 11, "axes.labelsize": 10, "legend.frameon": False})


def show_fig(fig, name):
    """Save a figure under reports/figures and display it."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")
    plt.show()


def tidy(df, d=4):
    """Round float columns only, for display."""
    return df.apply(lambda s: s.round(d) if pd.api.types.is_float_dtype(s) else s)


def money(x):
    """Signed dollar amount in millions."""
    return f"{'-' if x < 0 else ''}${abs(x) / 1e6:,.2f}M"


def usd(x):
    """Signed whole-dollar amount."""
    return f"{'-' if x < 0 else ''}${abs(x):,.0f}"


def pct(x):
    """Share as a percentage with one decimal."""
    return f"{x:.1%}"


def ratio(x):
    """Metric such as AUC with three decimals."""
    return f"{x:.3f}"


def mult(x):
    """Multiplier such as x1.25."""
    return f"x{x:.2f}"

In [ ]:
def find_data_file():
    """Return the raw data path, or stop with download instructions."""
    override = os.environ.get("CPL_DATA_FILE")
    if override:
        candidates = [Path(override)]
    else:
        candidates = sorted(p for p in DATA_DIR.glob("*") if p.name.lower().startswith("accepted_2007_to_2018q4")
                            and p.name.lower().endswith((".csv", ".csv.gz")))
    for p in candidates:
        if p.exists():
            return p
    print("Raw data file not found.\n"
          "1. Download accepted_2007_to_2018Q4.csv.gz from the Kaggle dataset 'Lending Club Loan Data'\n"
          "   (kaggle.com/datasets/wordsforthewise/lending-club).\n"
          f"2. Put it in {DATA_DIR} (keep the file name; the .csv.gz or the unzipped .csv both work).\n"
          "3. Re-run the notebook from the top.")
    raise FileNotFoundError("raw data file not found; see the instructions printed above")


def stable_hash(obj):
    """Short SHA-256 of a JSON-serialisable object, for change detection."""
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


DATA_FILE = find_data_file()
_stat = DATA_FILE.stat()
DATA_INFO = {"file": DATA_FILE.name, "size_bytes": _stat.st_size, "mtime": int(_stat.st_mtime)}
VERSIONS = {"python": sys.version.split()[0], **{m: importlib.import_module(m).__version__ for m in REQUIRED.values()}}

# everything that changes a result; each artifact hashes only the groups it depends on
SETTINGS = {"data": DATA_INFO, "demo_mode": DEMO_MODE, "seed": SEED, "schema": SCHEMA,
            "allowlist": sorted(ALLOWLIST), "model_features": MODEL_FEATURES, **CFG}
DEPS = {"loans_clean": ["data", "demo_mode", "seed", "schema", "load"]}
DEPS["splits"] = DEPS["loans_clean"] + ["cohort", "features", "allowlist", "model_features"]
DEPS["scorecard"] = DEPS["splits"] + ["scorecard"]
DEPS["lgbm"] = DEPS["splits"] + ["lgbm"]
DEPS["calibrators"] = DEPS["splits"] + ["scorecard", "lgbm"]

In [ ]:
MANIFEST_PATH = MODEL_DIR / "manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
manifest.update({"settings_hash": stable_hash(SETTINGS), "settings": SETTINGS, "versions": VERSIONS,
                 "data_file": DATA_INFO})
manifest.setdefault("artifacts", {})
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str))
print(f"data file: {DATA_FILE.name} ({DATA_INFO['size_bytes'] / 1e6:,.0f} MB) | settings hash: {manifest['settings_hash']}")


def write_json(path, obj):
    """Write JSON with full float precision so reloads are exact."""
    Path(path).write_text(json.dumps(obj, indent=1))


def load_or_train(name, train_fn, load_fn, save_fn, paths, model=True):
    """Load an artifact if its saved hash matches the current settings; otherwise build, save and reload it."""
    h = stable_hash({k: SETTINGS[k] for k in DEPS[name]})
    saved = manifest["artifacts"].get(name, {})
    fresh = saved.get("hash") == h and all(p.exists() for p in paths)
    if fresh and not (model and FORCE_RETRAIN):
        obj = load_fn()
        print(f"loaded {name} from {paths[0].relative_to(BASE).as_posix()}")
        return obj
    t0 = time.time()
    save_fn(train_fn())
    manifest["artifacts"][name] = {"hash": h, "paths": [p.relative_to(BASE).as_posix() for p in paths],
                                   "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str))
    print(f"{'trained' if model else 'built'} {name} in {time.time() - t0:.0f}s, saved")
    return load_fn()  # reload from disk so fresh and cached runs use identical objects

## Load and cache

I read only the columns in `SCHEMA`, with explicit dtypes, so the full file fits in Colab memory. The cleaned table is cached as parquet; later runs skip the slow CSV read.

In [ ]:
CLEAN_PATH = CACHE_DIR / "loans_clean.parquet"


def read_raw():
    """Read the needed columns, parse dates and term, drop summary rows, optionally sample."""
    dtypes = {SCHEMA[k]: (str if k in TEXT_COLS else "category" if k in CATEGORY_COLS else "float64")
              for k in SCHEMA}
    df = pd.read_csv(DATA_FILE, usecols=list(dtypes), dtype=dtypes, low_memory=False)
    df = df.rename(columns={v: k for k, v in SCHEMA.items()})
    df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
    df = df[df["issue_d"].notna()].copy()  # the Kaggle file ends with summary rows that have no issue date
    df["term_months"] = pd.to_numeric(df["term"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    if DEMO_MODE:
        df = df[df["term_months"] == CFG["cohort"]["term_months"]]
        df = df.sample(n=min(CFG["load"]["demo_rows"], len(df)), random_state=SEED)
    return df.sort_values(["issue_d", "id"]).reset_index(drop=True)


loans = load_or_train("loans_clean", read_raw, lambda: pd.read_parquet(CLEAN_PATH),
                      lambda d: d.to_parquet(CLEAN_PATH, index=False), [CLEAN_PATH], model=False)
print(f"loans_clean: {loans.shape[0]:,} rows x {loans.shape[1]} columns, "
      f"{loans.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory"
      + (" (DEMO_MODE sample)" if DEMO_MODE else ""))
del loans  # DuckDB reads the parquet file directly from here on

## Cohort and vintages

I keep 36-month loans only, so every loan had the same horizon to default. `bad = 1` for Charged Off and Default, `0` for Fully Paid.

Right-censoring matters: in recent vintages many loans are still running, and the ones that finish early are mostly the good ones, so dropping unresolved loans would make late vintages look safer than they are. I keep only issue months where at most 1% of loans are still unresolved.

In [ ]:
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE VIEW loans AS
    SELECT * REPLACE (CAST(issue_d AS DATE) AS issue_d, CAST(earliest_cr_line AS DATE) AS earliest_cr_line)
    FROM read_parquet('{CLEAN_PATH.as_posix().replace("'", "''")}')
""")
COH = CFG["cohort"]

MONTHLY_SQL = """
SELECT issue_d,
       COUNT(*) AS loans,
       AVG(CASE WHEN loan_status IN ('Current', 'In Grace Period') OR loan_status LIKE 'Late%'
                THEN 1.0 ELSE 0.0 END) AS unresolved_share
FROM loans
WHERE term_months = $term AND issue_d >= CAST($start AS DATE)
GROUP BY issue_d
ORDER BY issue_d
"""
monthly = con.execute(MONTHLY_SQL, {"term": COH["term_months"], "start": COH["start"]}).df()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(monthly["issue_d"], monthly["unresolved_share"], color=COLORS["scorecard"])
ax.axhline(COH["max_unresolved_share"], color=COLORS["baseline"], ls="--", label="maturity threshold")
ax.axvline(pd.Timestamp(COH["test_end"]), color=COLORS["lgbm"], ls=":", label="end of test window")
ax.set_yscale("symlog", linthresh=0.01)
ax.set(title="Share of 36-month loans still unresolved, by issue month", xlabel="issue month",
       ylabel="unresolved share (symlog)")
ax.legend()
show_fig(fig, "unresolved_share_by_month")

In [ ]:
COHORT_SQL = """
WITH base AS (
    SELECT * FROM loans
    WHERE term_months = $term
      AND issue_d BETWEEN CAST($start AS DATE) AND CAST($test_end AS DATE)
),
monthly AS (
    SELECT issue_d,
           AVG(CASE WHEN loan_status IN ('Current', 'In Grace Period') OR loan_status LIKE 'Late%'
                    THEN 1.0 ELSE 0.0 END) AS unresolved_share
    FROM base
    GROUP BY issue_d
)
SELECT b.*,
       CASE WHEN b.loan_status IN ('Charged Off', 'Default') THEN 1 ELSE 0 END AS bad,
       CASE WHEN b.issue_d <= CAST($train_end AS DATE) THEN 'train'
            WHEN b.issue_d <= CAST($valid_end AS DATE) THEN 'valid'
            ELSE 'test' END AS split
FROM base b
JOIN monthly m USING (issue_d)
WHERE m.unresolved_share <= $max_unresolved
  AND b.loan_status IN ('Fully Paid', 'Charged Off', 'Default')
ORDER BY b.issue_d, b.id
"""
cohort_params = {"term": COH["term_months"], "start": COH["start"], "test_end": COH["test_end"],
                 "train_end": COH["train_end"], "valid_end": COH["valid_end"],
                 "max_unresolved": COH["max_unresolved_share"]}
cohort = con.execute(COHORT_SQL, cohort_params).df()
con.register("cohort", cohort)

in_window = monthly[(monthly["issue_d"] >= pd.Timestamp(COH["start"])) & (monthly["issue_d"] <= pd.Timestamp(COH["test_end"]))]
dropped = in_window[in_window["unresolved_share"] > COH["max_unresolved_share"]]
print(f"maturity filter dropped {len(dropped)} of {len(in_window)} issue months in the cohort window")

split_summary = con.execute("""
    SELECT split, COUNT(*) AS loans, MIN(issue_d) AS first_issue, MAX(issue_d) AS last_issue, AVG(bad) AS bad_rate
    FROM cohort GROUP BY split
    ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'valid' THEN 2 ELSE 3 END
""").df()
display(tidy(split_summary))
print("note: the test vintage behaves differently from train (different bad rate and mix). "
      "That drift is deliberate: it is what an out-of-time test is for, and PSI below measures it.")

In [ ]:
VINTAGE_SQL = """
WITH q AS (
    SELECT date_trunc('quarter', issue_d) AS issue_q,
           COUNT(*) AS loans, SUM(bad) AS bads, AVG(bad) AS bad_rate, AVG(funded_amnt) AS avg_funded
    FROM cohort
    GROUP BY 1
)
SELECT issue_q, loans, bad_rate, avg_funded,
       loans * 1.0 / LAG(loans) OVER (ORDER BY issue_q) - 1  AS volume_qoq,
       bad_rate - LAG(bad_rate) OVER (ORDER BY issue_q)      AS bad_rate_qoq,
       SUM(bads) OVER w * 1.0 / SUM(loans) OVER w            AS bad_rate_rolling_4q
FROM q
WINDOW w AS (ORDER BY issue_q ROWS BETWEEN 3 PRECEDING AND CURRENT ROW)
ORDER BY issue_q
"""
vintage = con.execute(VINTAGE_SQL).df()
display(tidy(vintage))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(vintage["issue_q"], vintage["loans"], width=60, color=COLORS["bar"], label="loans (left)")
ax.set(title="Volume and bad rate by issue quarter", xlabel="issue quarter", ylabel="loans")
ax2 = ax.twinx()
ax2.plot(vintage["issue_q"], vintage["bad_rate"], color=COLORS["lgbm"], marker="o", label="bad rate (right)")
ax2.plot(vintage["issue_q"], vintage["bad_rate_rolling_4q"], color=COLORS["baseline"], ls="--",
         label="rolling 4-quarter bad rate (right)")
ax2.set_ylabel("bad rate")
ax2.grid(False)
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.88))
show_fig(fig, "vintage_volume_bad_rate")

## Features

Two SQL window features compare each applicant with peers:
- `inc_pctile_peer`: income percentile among loans with the same purpose in the same year. It uses same-year peers, but never their outcomes.
- `dti_vs_trailing_peer`: DTI minus the average DTI of the previous 1,000 same-purpose loans. The frame ends one row before the current loan, so there is no look-ahead.

The rest are simple transforms (average FICO, credit history length, logs, ratios, employment years) plus caps on thin tails. Missing values are filled with medians from **train only**.

In [ ]:
FEAT = CFG["features"]
FEATURE_SQL = """
WITH c AS (
    SELECT * REPLACE (LEAST(GREATEST(dti, $dti_lo), $dti_hi) AS dti) FROM cohort
)
SELECT c.*,
       (fico_range_low + fico_range_high) / 2.0              AS fico_avg,
       date_diff('day', earliest_cr_line, issue_d) / 365.25  AS credit_history_years,
       ln(1 + annual_inc)                                    AS log_income,
       LEAST(loan_amnt / NULLIF(annual_inc, 0), $lti_cap)    AS loan_to_income,
       ln(1 + revol_bal)                                     AS log_revol_bal,
       open_acc / NULLIF(total_acc, 0)                       AS open_acc_ratio,
       PERCENT_RANK() OVER (PARTITION BY purpose, year(issue_d) ORDER BY annual_inc) AS inc_pctile_peer,
       dti - AVG(dti) OVER (PARTITION BY purpose ORDER BY issue_d, id
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING)     AS dti_vs_trailing_peer
FROM c
ORDER BY issue_d, id
"""


def emp_years(s):
    """'10+ years' -> 10, '< 1 year' -> 0, 'n years' -> n, missing stays missing."""
    s = s.astype("string")
    years = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")
    return years.where(~s.str.contains("<", na=False), 0.0).astype(float)

In [ ]:
SPLIT_PATHS = {s: CACHE_DIR / f"{s}.parquet" for s in SPLITS}
MEDIANS_PATH = CACHE_DIR / "feature_medians.json"


def build_splits():
    """Window and derived features, caps, train-only imputation; one frame per split."""
    sql = FEATURE_SQL.format(window=int(FEAT["dti_peer_window"]))
    f = con.execute(sql, {"dti_lo": FEAT["dti_clip"][0], "dti_hi": FEAT["dti_clip"][1],
                          "lti_cap": FEAT["loan_to_income_cap"]}).df()
    f["emp_length_years"] = emp_years(f["emp_length"])
    f["emp_length_missing"] = f["emp_length_years"].isna().astype(int)
    for col, cap in FEAT["caps"].items():
        f[col] = f[col].clip(upper=cap)
    f["revol_util"] = f["revol_util"].clip(*FEAT["revol_util_clip"])
    for c in CAT_FEATURES:
        f[c] = f[c].astype("string").fillna("MISSING").astype(str)
    medians = f.loc[f["split"] == "train", NUM_FEATURES].median()  # train only: no information from later vintages
    f[NUM_FEATURES] = f[NUM_FEATURES].fillna(medians)
    keep = ["id", "issue_d", "split", "bad"] + ECON_COLS + MODEL_FEATURES
    frames = {s: f.loc[f["split"] == s, keep].reset_index(drop=True) for s in SPLITS}
    return frames, {k: float(v) for k, v in medians.items()}


def save_splits(obj):
    frames, medians = obj
    for s in SPLITS:
        frames[s].to_parquet(SPLIT_PATHS[s], index=False)
    write_json(MEDIANS_PATH, medians)


def load_splits():
    return {s: pd.read_parquet(SPLIT_PATHS[s]) for s in SPLITS}


frames = load_or_train("splits", build_splits, load_splits, save_splits,
                       list(SPLIT_PATHS.values()) + [MEDIANS_PATH], model=False)
train, valid, test = frames["train"], frames["valid"], frames["test"]
print({s: len(frames[s]) for s in SPLITS})

**Why `int_rate`, `grade` and `installment` are banned as features.** `grade` and `int_rate` are the incumbent lender's own risk price: using them would mean copying their model, not building one. `installment` looks innocent, but at a fixed 36-month term `installment / loan_amnt` is a deterministic function of `int_rate`, so it would smuggle the grade back in. The check below shows that relationship on train, then runs the leakage assertions.

In [ ]:
ratio_corr = np.corrcoef(train["installment"] / train["funded_amnt"], train["int_rate"])[0, 1]
print(f"[train] correlation of installment/funded_amnt with int_rate: {ratio_corr:.4f}")

assert_features_ok(MODEL_FEATURES)
for s in SPLITS:
    assert not set(frames[s].columns[frames[s].columns.isin(MODEL_FEATURES)]) & set(BANNED)
    assert frames[s][MODEL_FEATURES].isna().sum().sum() == 0, f"missing values left in {s}"
assert train["issue_d"].max() < valid["issue_d"].min() < test["issue_d"].min(), "splits overlap in time"
assert valid["issue_d"].max() < test["issue_d"].min(), "valid overlaps test"
assert len(set(train["id"]) & set(test["id"])) == 0 and len(set(valid["id"]) & set(test["id"])) == 0
assert train["issue_d"].min() >= pd.Timestamp(COH["start"]) and test["issue_d"].max() <= pd.Timestamp(COH["test_end"])
print("all leakage checks passed")

## Quick look at the signal

Before modelling, bad rates by FICO band, DTI decile and purpose on **train**. If these do not move, no model will help.

In [ ]:
con.register("train_df", train)
fico_bands = con.execute("""
    SELECT CASE WHEN fico_avg < 680 THEN '<680' WHEN fico_avg < 700 THEN '680-699'
                WHEN fico_avg < 720 THEN '700-719' WHEN fico_avg < 740 THEN '720-739'
                WHEN fico_avg < 760 THEN '740-759' ELSE '760+' END AS fico_band,
           MIN(fico_avg) AS lo, COUNT(*) AS loans, AVG(bad) AS bad_rate
    FROM train_df GROUP BY 1 ORDER BY lo
""").df()
dti_deciles = con.execute("""
    WITH d AS (SELECT bad, dti, NTILE(10) OVER (ORDER BY dti, id) AS dti_decile FROM train_df)
    SELECT dti_decile, MIN(dti) AS dti_from, MAX(dti) AS dti_to, COUNT(*) AS loans, AVG(bad) AS bad_rate
    FROM d GROUP BY 1 ORDER BY 1
""").df()
purpose_rates = con.execute("""
    SELECT purpose, COUNT(*) AS loans, AVG(bad) AS bad_rate
    FROM train_df GROUP BY 1
    HAVING COUNT(*) >= 0.01 * (SELECT COUNT(*) FROM train_df)
    ORDER BY bad_rate DESC
""").df()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].bar(fico_bands["fico_band"], fico_bands["bad_rate"], color=COLORS["bar"])
axes[0].set(title="Bad rate by FICO band", xlabel="average FICO", ylabel="bad rate")
axes[1].bar(dti_deciles["dti_decile"].astype(str), dti_deciles["bad_rate"], color=COLORS["bar"])
axes[1].set(title="Bad rate by DTI decile", xlabel="DTI decile (1 = lowest)", ylabel="bad rate")
axes[2].barh(purpose_rates["purpose"], purpose_rates["bad_rate"], color=COLORS["bar"])
axes[2].set(title="Bad rate by purpose (>=1% of loans)", xlabel="bad rate")
axes[0].tick_params(axis="x", rotation=30)
show_fig(fig, "signal_fico_dti_purpose")

## Scorecard

My own Weight of Evidence implementation: quantile bins for numbers (small bins merged until each holds 5% of train, missing values in their own bin), rare categories grouped into `OTHER`. WoE of a bin is `ln(share of goods / share of bads)`, smoothed; IV sums `(share of goods - share of bads) * WoE` over bins. I keep features with IV between 0.02 and 0.5, and drop the weaker of any pair whose WoE columns correlate above 0.7.

Then a logistic regression on the WoE columns, with L2 strength chosen on **validation** AUC. Points use the standard scaling `Factor = PDO / ln(2)`, `Offset = BaseScore - Factor * ln(BaseOdds)`: a score of 600 means the base good:bad odds (train bad rate, rounded to a clean number) and every 40 points doubles them.

In [ ]:
def merge_small_bins(edges, v, min_count):
    """Remove interior edges until every numeric bin holds at least min_count rows."""
    edges = list(edges)
    while edges:
        counts = np.bincount(np.searchsorted(edges, v, side="left"), minlength=len(edges) + 1)
        i = int(np.argmin(counts))
        if counts[i] >= min_count:
            break
        if i == 0:
            edges.pop(0)
        elif i == len(counts) - 1:
            edges.pop(i - 1)
        else:  # merge into the smaller neighbour
            edges.pop(i - 1 if counts[i - 1] <= counts[i + 1] else i)
    return edges


def assign_bins(x, spec):
    """Bin position of each value under a saved spec; len(labels) means a neutral fallback bin."""
    k = len(spec["labels"])
    if spec["type"] == "numeric":
        v = pd.to_numeric(x, errors="coerce").to_numpy(dtype=float)
        idx = np.searchsorted(spec["edges"], v, side="left")
        miss_pos = spec["labels"].index("MISSING") if "MISSING" in spec["labels"] else k
        return np.where(np.isnan(v), miss_pos, idx)
    pos = {lab: i for i, lab in enumerate(spec["labels"])}
    other = pos.get("OTHER", k)
    return x.astype(str).map(lambda c: pos.get(c, other)).to_numpy(dtype=int)

In [ ]:
def woe_bins(x, y, max_bins=10, min_share=0.05, smooth=0.5):
    """Bin one feature, then return its spec with counts, bad rate, WoE per bin and total IV."""
    y = np.asarray(y, dtype=float)
    if pd.api.types.is_numeric_dtype(x):
        v = x.to_numpy(dtype=float)
        ok = ~np.isnan(v)
        qs = np.quantile(v[ok], np.linspace(0, 1, max_bins + 1)[1:-1])
        edges = merge_small_bins(np.unique(qs), v[ok], min_share * ok.sum())
        bounds = [-np.inf] + edges + [np.inf]
        labels = [f"({bounds[i]:.4g}, {bounds[i + 1]:.4g}]" for i in range(len(edges) + 1)]
        labels += ["MISSING"] if (~ok).any() else []
        spec = {"type": "numeric", "edges": [float(e) for e in edges], "labels": labels}
    else:
        s = x.astype(str)
        share = s.value_counts(normalize=True)
        levels = sorted(share[share >= min_share].index)
        labels = levels + (["OTHER"] if (~s.isin(levels)).any() else [])
        spec = {"type": "categorical", "labels": labels}
    ids = assign_bins(x, spec)
    k = len(spec["labels"])
    n = np.bincount(ids, minlength=k).astype(float)
    bads = np.bincount(ids, weights=y, minlength=k)
    goods = n - bads
    dist_g = (goods + smooth) / (goods.sum() + smooth * k)
    dist_b = (bads + smooth) / (bads.sum() + smooth * k)
    woe = np.log(dist_g / dist_b)
    spec.update({"n": n.tolist(), "bad_rate": (bads / np.maximum(n, 1)).tolist(), "woe": woe.tolist(),
                 "iv": float(np.sum((dist_g - dist_b) * woe))})
    return spec

In [ ]:
SC = CFG["scorecard"]
WOE_PATH, SCORECARD_PATH = MODEL_DIR / "woe_tables.json", MODEL_DIR / "scorecard.json"


def woe_frame(df, specs, feats):
    """WoE-encoded copy of the given features; unseen values get WoE 0."""
    return pd.DataFrame({f: np.append(specs[f]["woe"], 0.0)[assign_bins(df[f], specs[f])] for f in feats},
                        index=df.index)


def select_features(specs, w_train):
    """IV window first, then drop the weaker feature of each highly correlated pair."""
    iv = pd.Series({f: s["iv"] for f, s in specs.items()}).sort_values(ascending=False)
    candidates = [f for f in iv.index if SC["iv_min"] <= iv[f] <= SC["iv_max"]]
    corr = w_train[candidates].corr().abs()
    selected = []
    for f in candidates:  # strongest first, so the weaker member of a correlated pair is the one dropped
        if all(corr.loc[f, g] <= SC["max_woe_corr"] for g in selected):
            selected.append(f)
    return selected

In [ ]:
def train_scorecard():
    """WoE bins, feature selection, C on validation AUC, then points scaling."""
    y = train["bad"].to_numpy()
    specs = {f: woe_bins(train[f], y, SC["max_bins"], SC["min_share"], SC["woe_smoothing"]) for f in MODEL_FEATURES}
    w_train = woe_frame(train, specs, MODEL_FEATURES)
    selected = select_features(specs, w_train)
    assert_features_ok(selected)
    w_valid = woe_frame(valid, specs, selected)
    c_results = []
    for C in SC["C_grid"]:
        m = LogisticRegression(C=C, max_iter=2000).fit(w_train[selected], y)
        c_results.append([C, float(roc_auc_score(valid["bad"], m.predict_proba(w_valid)[:, 1]))])
    best_c = max(c_results, key=lambda r: r[1])[0]
    m = LogisticRegression(C=best_c, max_iter=2000).fit(w_train[selected], y)
    coefs, b0 = dict(zip(selected, m.coef_[0].tolist())), float(m.intercept_[0])
    odds = (1 - y.mean()) / y.mean()
    base_odds = float(round(odds)) if odds >= 2 else round(odds * 2) / 2
    factor = SC["pdo"] / np.log(2)
    offset = SC["base_score"] - factor * np.log(base_odds)
    n_feat = len(selected)
    points = {}
    for f in selected:  # points per bin: the bin's share of offset + factor * ln(good odds)
        woe_ext = np.append(specs[f]["woe"], 0.0)
        pts = np.round(-(coefs[f] * woe_ext + b0 / n_feat) * factor + offset / n_feat)
        points[f] = pts.tolist()
    ref = {f: float(np.mean(np.asarray(points[f])[assign_bins(train[f], specs[f])])) for f in selected}
    return {"specs": specs, "selected": selected, "C_results": c_results, "C": best_c, "intercept": b0,
            "coefs": coefs, "scaling": {"base_score": SC["base_score"], "pdo": SC["pdo"], "base_odds": base_odds,
                                        "factor": float(factor), "offset": float(offset)},
            "points": points, "ref_points": ref}


def save_scorecard(sc):
    write_json(WOE_PATH, sc["specs"])
    write_json(SCORECARD_PATH, {k: v for k, v in sc.items() if k != "specs"})


def load_scorecard():
    return {**json.loads(SCORECARD_PATH.read_text()), "specs": json.loads(WOE_PATH.read_text())}


sc = load_or_train("scorecard", train_scorecard, load_scorecard, save_scorecard, [WOE_PATH, SCORECARD_PATH])

In [ ]:
iv_table = pd.DataFrame({"iv": {f: s["iv"] for f, s in sc["specs"].items()}}).sort_values("iv", ascending=False)
iv_table["bins"] = [len(sc["specs"][f]["labels"]) for f in iv_table.index]
iv_table["kept"] = iv_table.index.isin(sc["selected"])
print("[train] information value per feature (kept = inside the IV window and not too correlated)")
display(tidy(iv_table))
print("[valid] AUC by C:", {c: round(a, 4) for c, a in sc["C_results"]}, f"-> chosen C = {sc['C']}")
s = sc["scaling"]
print(f"scaling: base score {s['base_score']} at good:bad odds {s['base_odds']:g}:1, PDO {s['pdo']}, "
      f"factor {s['factor']:.3f}, offset {s['offset']:.3f}")

rows = []
for f in sc["selected"]:
    spec = sc["specs"][f]
    for lab, n, br, w, p in zip(spec["labels"], spec["n"], spec["bad_rate"], spec["woe"], sc["points"][f]):
        rows.append({"feature": f, "bin": lab, "train_loans": int(n), "bad_rate": br, "woe": w, "points": int(p)})
points_table = pd.DataFrame(rows)
print("scorecard points per bin (higher points = lower risk)")
display(tidy(points_table))


def scorecard_apply(df):
    """Scorecard PD, total points and points per feature, computed from the saved JSON only."""
    logit = np.full(len(df), sc["intercept"])
    pts = {}
    for f in sc["selected"]:
        idx = assign_bins(df[f], sc["specs"][f])
        logit += sc["coefs"][f] * np.append(sc["specs"][f]["woe"], 0.0)[idx]
        pts[f] = np.asarray(sc["points"][f])[idx]
    pts = pd.DataFrame(pts, index=df.index)
    return 1 / (1 + np.exp(-logit)), pts.sum(axis=1).to_numpy(), pts


for df in (train, valid, test):
    df["pd_sc_raw"], df["score_sc"], _ = scorecard_apply(df)
print("mean score by split:", {s: round(float(frames[s]["score_sc"].mean()), 1) for s in SPLITS})

top3 = [f for f in iv_table.index if f in sc["selected"] and sc["specs"][f]["type"] == "numeric"][:3]
fig, axes = plt.subplots(1, len(top3), figsize=(4.5 * len(top3), 3.6), squeeze=False)
for ax, f in zip(axes[0], top3):
    spec = sc["specs"][f]
    ax.bar(range(len(spec["woe"])), spec["woe"], color=COLORS["scorecard"])
    ax.set_xticks(range(len(spec["labels"])), spec["labels"], rotation=60, ha="right", fontsize=7)
    ax.set(title=f"WoE by bin: {f}", ylabel="WoE (higher = safer)")
show_fig(fig, "woe_top_features")

## LightGBM challenger

Gradient boosting on the raw (not WoE) features. I add monotone constraints only where the direction of risk is not in doubt, for example higher FICO can never raise predicted risk; this keeps the model explainable and stops it learning noise. Parameters are fixed at sensible values; only the number of trees is chosen, by early stopping on **validation** AUC.

In [ ]:
LG = CFG["lgbm"]
LGBM_PATH = MODEL_DIR / "lgbm.txt"
CAT_LEVELS = {c: sorted(train[c].unique()) for c in CAT_FEATURES}  # fixed category codes across splits


def model_frame(df):
    """LightGBM feature matrix with categories pinned to the train levels."""
    X = df[MODEL_FEATURES].copy()
    for c in CAT_FEATURES:
        X[c] = pd.Categorical(X[c], categories=CAT_LEVELS[c])
    assert_features_ok(X.columns)
    return X


def train_lgbm():
    """Monotone-constrained LightGBM with early stopping on validation AUC."""
    params = {**LG["params"], "monotone_constraints": [LG["monotone"].get(f, 0) for f in MODEL_FEATURES]}
    dtrain = lgb.Dataset(model_frame(train), train["bad"])
    dvalid = lgb.Dataset(model_frame(valid), valid["bad"], reference=dtrain)
    return lgb.train(params, dtrain, num_boost_round=LG["demo_rounds"] if DEMO_MODE else LG["num_rounds"],
                     valid_sets=[dvalid], valid_names=["valid"],
                     callbacks=[lgb.early_stopping(LG["early_stopping"], verbose=False)])


booster = load_or_train("lgbm", train_lgbm, lambda: lgb.Booster(model_file=str(LGBM_PATH)),
                        lambda b: b.save_model(str(LGBM_PATH), num_iteration=b.best_iteration), [LGBM_PATH])
print(f"[valid] trees kept after early stopping: {booster.num_trees()}")

for df in (train, valid, test):
    df["pd_lgb_raw"] = booster.predict(model_frame(df))

gain = pd.Series(booster.feature_importance("gain"), index=booster.feature_name()).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(gain.index, gain / gain.sum(), color=COLORS["lgbm"])
ax.set(title="LightGBM feature importance (share of total gain)", xlabel="share of gain")
show_fig(fig, "lgbm_gain_importance")

## Calibration

The profit layer multiplies PD by dollars, so a PD of 10% has to mean one loan in ten defaults. Raw model scores rank well but are not guaranteed to be probabilities, and the test vintage is riskier than train. I fit isotonic regression on **validation** predictions, save its knots as JSON and apply them with `np.interp`. This is also why I do not use SMOTE or any resampling: it would shift the base rate and break exactly this property.

In [ ]:
CAL_PATH = MODEL_DIR / "calibrators.json"


def train_calibrators():
    """Isotonic knots per model, fitted on validation predictions."""
    out = {}
    for name, col in [("scorecard", "pd_sc_raw"), ("lgbm", "pd_lgb_raw")]:
        iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(valid[col], valid["bad"])
        out[name] = {"x": iso.X_thresholds_.tolist(), "y": iso.y_thresholds_.tolist()}
    return out


calibrators = load_or_train("calibrators", train_calibrators, lambda: json.loads(CAL_PATH.read_text()),
                            lambda c: write_json(CAL_PATH, c), [CAL_PATH])


def calibrate(p, name):
    """Apply saved isotonic knots; np.interp clips outside the fitted range like the original model."""
    return np.interp(p, calibrators[name]["x"], calibrators[name]["y"])


for df in (train, valid, test):
    df["pd_sc"] = calibrate(df["pd_sc_raw"], "scorecard")
    df["pd_lgb"] = calibrate(df["pd_lgb_raw"], "lgbm")

In [ ]:
def decile_calibration(df, col):
    """Mean predicted PD against actual bad rate by decile of predicted PD."""
    dec = pd.qcut(df[col].rank(method="first"), 10, labels=False) + 1
    return df.groupby(dec).agg(mean_pd=(col, "mean"), bad_rate=("bad", "mean"), loans=("bad", "size"))


cal_sc, cal_lgb = decile_calibration(test, "pd_sc"), decile_calibration(test, "pd_lgb")
cal_table = pd.concat({"scorecard": cal_sc, "lgbm": cal_lgb}, axis=1)
cal_table.index.name = "decile"
print("[test] decile calibration (calibrators fitted on valid)")
display(tidy(cal_table))

brier = pd.DataFrame({m: {"raw": brier_score_loss(test["bad"], test[r]), "calibrated": brier_score_loss(test["bad"], test[c])}
                      for m, r, c in [("scorecard", "pd_sc_raw", "pd_sc"), ("lgbm", "pd_lgb_raw", "pd_lgb")]})
print("[test] Brier score before and after calibration (lower is better)")
display(tidy(brier, 5))
print(f"[test] mean calibrated PD: scorecard {test['pd_sc'].mean():.4f}, LightGBM {test['pd_lgb'].mean():.4f} "
      f"| actual bad rate {test['bad'].mean():.4f}")

fig, ax = plt.subplots(figsize=(5, 4.5))
lim = max(cal_table.max().max(), 0.05)
ax.plot([0, lim], [0, lim], color=COLORS["baseline"], ls="--", label="perfect calibration")
ax.plot(cal_sc["mean_pd"], cal_sc["bad_rate"], marker="o", color=COLORS["scorecard"], label="scorecard")
ax.plot(cal_lgb["mean_pd"], cal_lgb["bad_rate"], marker="s", color=COLORS["lgbm"], label="LightGBM")
ax.set(title="Calibration on test (deciles)", xlabel="mean predicted PD", ylabel="actual bad rate")
ax.legend()
show_fig(fig, "calibration_test")

## Out-of-time evaluation

Ranking metrics (AUC, KS, Gini) use the raw scores, because isotonic calibration adds ties without changing order; Brier uses the calibrated PDs. The incumbent baseline is `int_rate` used directly as a score: it is the price the original lender set with information I do not have, so it is a hard benchmark.

In [ ]:
EV_ = CFG["eval"]


def ks_stat(y, s):
    """Largest gap between the cumulative score distributions of bads and goods."""
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr - fpr))


rows = []
for split, df in frames.items():
    for model, score, pd_col in [("scorecard", "pd_sc_raw", "pd_sc"), ("lgbm", "pd_lgb_raw", "pd_lgb"),
                                 ("int_rate baseline", "int_rate", None)]:
        auc = roc_auc_score(df["bad"], df[score])
        rows.append({"split": split, "model": model, "AUC": auc, "KS": ks_stat(df["bad"], df[score]),
                     "Gini": 2 * auc - 1, "Brier": brier_score_loss(df["bad"], df[pd_col]) if pd_col else np.nan})
metrics = pd.DataFrame(rows).set_index(["split", "model"])
print("C, early stopping and calibration were chosen on valid; test was not used for any choice")
display(tidy(metrics))

test_auc = metrics.loc["test"]["AUC"]
for model in ["scorecard", "lgbm"]:
    if test_auc[model] > EV_["leakage_auc"]:
        print("!" * 90 + f"\nWARNING: [test] {model} AUC {test_auc[model]:.4f} is above {EV_['leakage_auc']}. "
              "An origination-only model on this data almost certainly cannot do this: check for leakage.\n" + "!" * 90)
gap = test_auc["lgbm"] - test_auc["int_rate baseline"]
print(f"[test] LightGBM AUC minus int_rate AUC: {gap:+.4f} "
      + ("(the incumbent's price still ranks risk better)" if gap < 0 else "(the model ranks at least as well as the price)"))

In [ ]:
def bootstrap_auc(y, scores, n_boot, seed):
    """Paired bootstrap AUCs via resampling weights; one sort per score keeps this fast."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=float)
    prep = []
    for s in scores:
        order = np.argsort(s, kind="mergesort")
        s_sorted = np.asarray(s)[order]
        prep.append((order, y[order], np.flatnonzero(np.r_[True, s_sorted[1:] != s_sorted[:-1]])))
    out = np.empty((n_boot, len(scores)))
    for b in range(n_boot):
        w = np.bincount(rng.integers(0, len(y), len(y)), minlength=len(y)).astype(float)
        for j, (order, ys, starts) in enumerate(prep):
            ws = w[order]
            pos, neg = np.add.reduceat(ws * ys, starts), np.add.reduceat(ws * (1 - ys), starts)
            below = np.cumsum(neg) - neg  # weighted goods scored strictly lower; ties count half
            out[b, j] = np.sum(pos * (below + 0.5 * neg)) / (pos.sum() * neg.sum())
    return out


boot = bootstrap_auc(test["bad"], [test["pd_sc_raw"].to_numpy(), test["pd_lgb_raw"].to_numpy()],
                     EV_["n_bootstrap"], SEED)
ci = lambda a: np.percentile(a, [2.5, 97.5])
ci_sc, ci_lgb, ci_diff = ci(boot[:, 0]), ci(boot[:, 1]), ci(boot[:, 1] - boot[:, 0])
print(f"[test] AUC 95% bootstrap interval ({EV_['n_bootstrap']} resamples): "
      f"scorecard {ci_sc[0]:.4f} to {ci_sc[1]:.4f}, LightGBM {ci_lgb[0]:.4f} to {ci_lgb[1]:.4f}")
gap_meaningful = ci_diff[0] > 0 or ci_diff[1] < 0
print(f"[test] LightGBM minus scorecard: {ci_diff[0]:+.4f} to {ci_diff[1]:+.4f} -> "
      + ("the gap is meaningful (interval excludes zero)" if gap_meaningful else "the gap is not meaningful (interval includes zero)"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for name, col, color in [("scorecard", "pd_sc_raw", COLORS["scorecard"]), ("LightGBM", "pd_lgb_raw", COLORS["lgbm"]),
                         ("int_rate baseline", "int_rate", COLORS["baseline"])]:
    fpr, tpr, _ = roc_curve(test["bad"], test[col])
    axes[0].plot(fpr, tpr, color=color, label=f"{name} (AUC {roc_auc_score(test['bad'], test[col]):.3f})")
axes[0].plot([0, 1], [0, 1], color="black", lw=0.6, ls=":")
axes[0].set(title="ROC on test", xlabel="false positive rate (goods flagged)", ylabel="true positive rate (bads caught)")
axes[0].legend(fontsize=8)

order = np.argsort(-test["pd_lgb_raw"].to_numpy())
yb = test["bad"].to_numpy()[order]
cum_bad, cum_good = np.cumsum(yb) / yb.sum(), np.cumsum(1 - yb) / (1 - yb).sum()
share = np.arange(1, len(yb) + 1) / len(yb)
k = int(np.argmax(cum_bad - cum_good))
axes[1].plot(share, cum_bad, color=COLORS["lgbm"], label="bads (cumulative share)")
axes[1].plot(share, cum_good, color=COLORS["baseline"], label="goods (cumulative share)")
axes[1].vlines(share[k], cum_good[k], cum_bad[k], color="black", ls="--", label=f"KS = {cum_bad[k] - cum_good[k]:.3f}")
axes[1].set(title="KS on test, LightGBM", xlabel="share of loans, riskiest first", ylabel="cumulative share")
axes[1].legend(fontsize=8)
show_fig(fig, "roc_ks_test")

con.register("test_df", test)
LIFT_SQL = """
WITH s AS (
    SELECT bad, NTILE(10) OVER (ORDER BY {col} DESC, id) AS decile FROM test_df
), d AS (
    SELECT decile, COUNT(*) AS loans, SUM(bad) AS bads, AVG(bad) AS bad_rate FROM s GROUP BY decile
)
SELECT decile, loans, bad_rate,
       bad_rate / (SUM(bads) OVER () * 1.0 / SUM(loans) OVER ())   AS lift,
       SUM(bads) OVER (ORDER BY decile) * 1.0 / SUM(bads) OVER ()  AS cum_bad_capture
FROM d ORDER BY decile
"""
lift = pd.concat({name: con.execute(LIFT_SQL.format(col=col)).df().set_index("decile")
                  for name, col in [("scorecard", "pd_sc_raw"), ("lgbm", "pd_lgb_raw")]}, axis=1)
print("[test] lift by decile (1 = riskiest tenth of loans)")
display(tidy(lift))

In [ ]:
def psi(expected, actual, bins=10):
    """Population stability index of actual against expected, with expected's quantile bins or levels."""
    expected, actual = pd.Series(expected), pd.Series(actual)
    if pd.api.types.is_numeric_dtype(expected):
        edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)[1:-1]))
        e = np.bincount(np.searchsorted(edges, expected, side="left"), minlength=len(edges) + 1)
        a = np.bincount(np.searchsorted(edges, actual, side="left"), minlength=len(edges) + 1)
    else:
        levels = sorted(set(expected) | set(actual))
        e = expected.value_counts().reindex(levels, fill_value=0).to_numpy()
        a = actual.value_counts().reindex(levels, fill_value=0).to_numpy()
    e = np.clip(e / e.sum(), 1e-6, None)
    a = np.clip(a / a.sum(), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))


def psi_band(v):
    """Usual reading: below 0.1 stable, 0.1 to 0.25 moderate shift, above 0.25 major shift."""
    return "stable" if v < 0.1 else "moderate shift" if v < 0.25 else "major shift"


score_psi = pd.DataFrame([{"score": name, "train->valid": psi(train[col], valid[col], EV_["psi_bins"]),
                           "train->test": psi(train[col], test[col], EV_["psi_bins"])}
                          for name, col in [("scorecard points", "score_sc"), ("LightGBM PD", "pd_lgb_raw")]])
score_psi["band (train->test)"] = score_psi["train->test"].map(psi_band)
display(tidy(score_psi))

top8 = gain.sort_values(ascending=False).index[:8]
feat_psi = pd.DataFrame({"psi train->test": [psi(train[f], test[f], EV_["psi_bins"]) for f in top8]}, index=top8)
feat_psi["band"] = feat_psi["psi train->test"].map(psi_band)
print("PSI of the top 8 LightGBM features, train -> test")
display(tidy(feat_psi))

## Reason codes

A lender must tell a declined applicant why. For the five riskiest test applicants I list the top three adverse factors from each model: for the scorecard, points lost against the average points for that feature in train; for LightGBM, the largest positive contributions to the log-odds of default from `pred_contrib=True`.

In [ ]:
riskiest = test.nlargest(5, "pd_lgb").copy()
_, _, pts = scorecard_apply(riskiest)
lost = pd.DataFrame({f: sc["ref_points"][f] - pts[f] for f in sc["selected"]}, index=riskiest.index)
contrib = pd.DataFrame(booster.predict(model_frame(riskiest), pred_contrib=True)[:, :-1],
                       columns=MODEL_FEATURES, index=riskiest.index)


def fmt_value(v):
    """Short display of a feature value."""
    return f"{v:.3g}" if isinstance(v, (int, float, np.floating, np.integer)) else str(v)


reasons = pd.DataFrame({
    "id": riskiest["id"],
    "pd_lgbm": riskiest["pd_lgb"].round(3),
    "score": riskiest["score_sc"].astype(int),
    "scorecard: points lost vs average": [
        "; ".join(f"{f}={fmt_value(riskiest.at[i, f])} (-{lost.at[i, f]:.0f})" for f in lost.loc[i].nlargest(3).index)
        for i in riskiest.index],
    "LightGBM: log-odds added": [
        "; ".join(f"{f}={fmt_value(riskiest.at[i, f])} (+{contrib.at[i, f]:.2f})" for f in contrib.loc[i].nlargest(3).index)
        for i in riskiest.index],
})
print("[test] top adverse factors for the five highest-PD applicants")
with pd.option_context("display.max_colwidth", None):
    display(reasons.reset_index(drop=True))

## Economics

Assumptions live in `CFG["economics"]`: cost of funds, an average-life factor that converts an annual rate into dollars over a 36-month amortising loan, and operating cost. Two parameters come from **train** cash flows:
- `realization`: realised interest on good loans divided by contractual interest (`installment * 36 - funded`), which captures prepayment.
- `bad_result`: net cash result per dollar funded on bad loans, including partial repayments and recoveries (net of collection fees, which go to the collection agency).

Expected profit per loan: `EV = funded * [(1 - PD) * realization * cont_yield + PD * bad_result] - funded * COF * avg_life - funded * opex`, with `cont_yield = (installment * 36 - funded) / funded`. Backtests use realised P&L from actual cash flows instead: interest income `total_rec_int + total_rec_late_fee`, gross profit `total_pymnt - collection_recovery_fee - funded_amnt`, credit loss = interest income - gross profit (so the lines reconcile exactly), and net profit = interest income - credit loss - funding cost - opex.

In [ ]:
ECON = CFG["economics"]
T = COH["term_months"]

sum_cols = train["total_rec_prncp"] + train["total_rec_int"] + train["total_rec_late_fee"] + train["recoveries"]
all_cf = pd.concat(frames.values())
match = (all_cf["total_pymnt"] - (all_cf["total_rec_prncp"] + all_cf["total_rec_int"]
                                  + all_cf["total_rec_late_fee"] + all_cf["recoveries"])).abs() <= 1.0
print(f"[all splits] total_pymnt equals principal + interest + late fees + recoveries (within $1) "
      f"for {match.mean():.2%} of loans")

for df in frames.values():
    df["cont_yield"] = (df["installment"] * T - df["funded_amnt"]) / df["funded_amnt"]

good, badl = train[train["bad"] == 0], train[train["bad"] == 1]
REALIZATION = float(good["total_rec_int"].sum() / (good["installment"] * T - good["funded_amnt"]).sum())
BAD_RESULT = float((badl["total_pymnt"] - badl["collection_recovery_fee"] - badl["funded_amnt"]).sum()
                   / badl["funded_amnt"].sum())
print(f"[train] realization = {REALIZATION:.4f} (share of contractual interest actually collected on good loans)")
print(f"[train] bad_result = {BAD_RESULT:.4f} (net cash per dollar funded on bad loans)")


def expected_value(df, pd_, cof=ECON["cof"], pd_mult=1.0, bad_mult=1.0):
    """Expected profit per loan in dollars, from the formula above."""
    p = np.clip(np.asarray(pd_) * pd_mult, 0.0, 1.0)
    f = df["funded_amnt"].to_numpy()
    return (f * ((1 - p) * REALIZATION * df["cont_yield"].to_numpy() + p * BAD_RESULT * bad_mult)
            - f * cof * ECON["avg_life_eq"] - f * ECON["opex_pct"])

In [ ]:
def realised_pnl(df):
    """Per-loan realised P&L lines from actual cash flows."""
    f = df["funded_amnt"]
    interest = df["total_rec_int"] + df["total_rec_late_fee"]
    gross = df["total_pymnt"] - df["collection_recovery_fee"] - f
    out = pd.DataFrame({"funded": f, "interest_income": interest, "credit_loss": interest - gross,
                        "funding_cost": f * ECON["cof"] * ECON["avg_life_eq"], "opex": f * ECON["opex_pct"]})
    out["net_profit"] = out["interest_income"] - out["credit_loss"] - out["funding_cost"] - out["opex"]
    return out


for df in frames.values():
    df[["interest_income", "credit_loss", "funding_cost", "opex", "net_profit"]] = realised_pnl(df).drop(columns="funded")


def pnl_summary(df, mask):
    """Portfolio P&L of the approved loans."""
    a = df[mask]
    return {"approval_rate": mask.mean(), "loans": int(mask.sum()), "funded": a["funded_amnt"].sum(),
            "interest_income": a["interest_income"].sum(), "credit_loss": a["credit_loss"].sum(),
            "funding_and_opex": (a["funding_cost"] + a["opex"]).sum(), "net_profit": a["net_profit"].sum(),
            "roi": a["net_profit"].sum() / max(a["funded_amnt"].sum(), 1.0), "bad_rate": a["bad"].mean()}


band_edges = np.quantile(valid["score_sc"], np.linspace(0, 1, ECON["n_risk_bands"] + 1)[1:-1])
test["risk_band"] = np.searchsorted(band_edges, test["score_sc"], side="right") + 1  # band 1 = lowest score = riskiest
bands = test.groupby("risk_band").agg(
    score_from=("score_sc", "min"), score_to=("score_sc", "max"), loans=("bad", "size"),
    predicted_pd=("pd_sc", "mean"), actual_bad_rate=("bad", "mean"), avg_int_rate=("int_rate", "mean"),
    interest_income=("interest_income", "sum"), credit_loss=("credit_loss", "sum"),
    net_profit=("net_profit", "sum"), funded=("funded_amnt", "sum"))
bands["roi"] = bands["net_profit"] / bands["funded"]
for c in ["interest_income", "credit_loss", "net_profit"]:
    bands[c] = (bands[c] / 1e6).round(3)
bands = bands.drop(columns="funded").rename(columns={c: c + "_$M" for c in ["interest_income", "credit_loss", "net_profit"]})
print(f"[test] realised P&L by scorecard risk band (band edges fixed on valid; band 1 = riskiest)")
display(tidy(bands))
print(f"loss-making bands on test: {[int(b) for b in bands.index[bands['net_profit_$M'] < 0]]}")

## Policy selection and backtest

Because interest income differs by loan, the profit-maximising rule is "approve when expected profit is positive", not a single PD cutoff: a riskier loan at a high rate can be worth more than a safer loan at a low rate. I compare four policies: approve all, the incumbent grade policy (approve grades up to a cutoff), and EV policies from each model that approve when `EV > hurdle`. The hurdle and the grade cutoff are chosen on **validation** by maximising realised net profit; then each policy is evaluated once on test.

In [ ]:
for df in (valid, test):
    df["ev_sc"] = expected_value(df, df["pd_sc"])
    df["ev_lgb"] = expected_value(df, df["pd_lgb"])


def best_cutoff(score, net, candidates):
    """Cutoff c maximising realised net profit when approving every loan with score > c."""
    score, net = np.asarray(score), np.asarray(net)
    order = np.argsort(-score, kind="mergesort")
    cum = np.r_[0.0, np.cumsum(net[order])]
    counts = np.searchsorted(-score[order], -np.asarray(candidates), side="left")  # loans with score > c
    i = int(np.argmax(cum[counts]))
    return float(candidates[i]), float(cum[counts][i])


HURDLES = {}
for name, col in [("scorecard", "ev_sc"), ("lgbm", "ev_lgb")]:
    ev = valid[col].to_numpy()
    cands = np.unique(np.r_[ev.min() - 1.0, 0.0, np.quantile(ev, np.linspace(0, 1, ECON["hurdle_quantiles"]))])
    HURDLES[name], _ = best_cutoff(ev, valid["net_profit"], cands)

GRADES = sorted(train["grade"].astype(str).unique())
grade_rank = {g: i for i, g in enumerate(GRADES)}
grade_profit = {g: valid.loc[valid["grade"].astype(str).map(grade_rank) <= grade_rank[g], "net_profit"].sum() for g in GRADES}
GRADE_CUT = max(grade_profit, key=grade_profit.get)
print(f"[valid] chosen hurdles: scorecard EV > {HURDLES['scorecard']:,.2f} $, LightGBM EV > {HURDLES['lgbm']:,.2f} $ "
      f"| grade policy: approve grades {GRADES[0]} to {GRADE_CUT}")


def policy_masks(df):
    """Approval decision of every policy for a frame."""
    return {"approve all": np.ones(len(df), dtype=bool),
            "grade policy": (df["grade"].astype(str).map(grade_rank) <= grade_rank[GRADE_CUT]).to_numpy(),
            "scorecard EV": (df["ev_sc"] > HURDLES["scorecard"]).to_numpy(),
            "LightGBM EV": (df["ev_lgb"] > HURDLES["lgbm"]).to_numpy()}

In [ ]:
def pnl_view(table):
    """P&L table for display, with dollar columns in millions."""
    out = table.astype(float).copy()
    usd_cols = ["funded", "interest_income", "credit_loss", "funding_and_opex", "net_profit"]
    out[usd_cols] = out[usd_cols] / 1e6
    return tidy(out.rename(columns={c: c + "_$M" for c in usd_cols}))


valid_pnl = pd.DataFrame({k: pnl_summary(valid, m) for k, m in policy_masks(valid).items()}).T
print("[valid] realised P&L by policy (used to choose)")
display(pnl_view(valid_pnl)[["approval_rate", "net_profit_$M", "roi", "bad_rate"]])

p_sc, p_lgb = valid_pnl.loc["scorecard EV", "net_profit"], valid_pnl.loc["LightGBM EV", "net_profit"]
CHAMP = "LightGBM EV" if p_lgb >= p_sc else "scorecard EV"
CHAMP_MODEL = "lgbm" if CHAMP == "LightGBM EV" else "scorecard"
CHAMP_PD, CHAMP_EV = ("pd_lgb", "ev_lgb") if CHAMP_MODEL == "lgbm" else ("pd_sc", "ev_sc")
close = abs(p_lgb - p_sc) <= ECON["explain_margin"] * max(abs(p_lgb), abs(p_sc))
print(f"[valid] champion: {CHAMP} (scorecard EV {money(p_sc)}, LightGBM EV {money(p_lgb)})")
if close:
    print("[valid] the two EV policies are within the explainability margin; the scorecard's fixed points "
          "per bin are easier to explain to a credit committee and a regulator, which can justify choosing it.")
if valid_pnl.loc["grade policy", "net_profit"] > max(p_sc, p_lgb):
    print("[valid] note: the incumbent grade policy beats both model policies on validation profit.")

test_masks = policy_masks(test)
test_pnl = pd.DataFrame({k: pnl_summary(test, m) for k, m in test_masks.items()}).T
print("[test] realised P&L by policy, evaluated once with hurdles and grade cutoff fixed on valid")
display(pnl_view(test_pnl))
print("Only approved loans have outcomes, so a policy here can only tighten approvals, never loosen them; "
      "the backtest cannot see applicants the original lender rejected.")

In [ ]:
base, pol = test_pnl.loc["approve all"], test_pnl.loc[CHAMP]
steps = {"interest income forgone": -(base["interest_income"] - pol["interest_income"]),
         "credit losses avoided": base["credit_loss"] - pol["credit_loss"],
         "funding and opex saved": base["funding_and_opex"] - pol["funding_and_opex"]}
net_change = pol["net_profit"] - base["net_profit"]
assert abs(sum(steps.values()) - net_change) < 1e-3 * max(1.0, abs(net_change)), "bridge does not reconcile"

labels = ["approve all"] + list(steps) + [CHAMP]
fig, ax = plt.subplots(figsize=(9, 4))
level = base["net_profit"]
ax.bar(0, level / 1e6, color=COLORS["baseline"])
for i, (k, v) in enumerate(steps.items(), start=1):
    ax.bar(i, v / 1e6, bottom=level / 1e6, color=COLORS["ev"] if v >= 0 else COLORS["lgbm"])
    level += v
ax.bar(len(labels) - 1, pol["net_profit"] / 1e6, color=COLORS["scorecard"])
ax.set_xticks(range(len(labels)), labels, rotation=15)
ax.axhline(0, color="black", lw=0.6)
ax.set(title=f"Test net profit bridge: approve all -> {CHAMP} (net change {money(net_change)})", ylabel="$ millions")
show_fig(fig, "profit_bridge_test")


def profit_curve(df, key, ascending):
    """Cumulative realised net profit when approving loans in ranking order."""
    order = np.argsort(df[key].to_numpy() if ascending else -df[key].to_numpy(), kind="mergesort")
    return np.arange(1, len(df) + 1) / len(df), np.cumsum(df["net_profit"].to_numpy()[order])


rankings = [("grade / int_rate", "int_rate", True, COLORS["baseline"]), ("scorecard PD", "pd_sc", True, COLORS["scorecard"]),
            ("LightGBM PD", "pd_lgb", True, COLORS["lgbm"]), ("LightGBM EV", "ev_lgb", False, COLORS["ev"])]
fig, ax = plt.subplots(figsize=(8, 4.5))
at_champ = {}
for name, key, asc, color in rankings:
    x, y = profit_curve(test, key, asc)
    ax.plot(x, y / 1e6, color=color, label=name)
    at_champ[name] = y[max(int(round(pol["approval_rate"] * len(x))) - 1, 0)]
ax.scatter([pol["approval_rate"]], [pol["net_profit"] / 1e6], color="black", zorder=5, label=f"{CHAMP} (chosen)")
ax.set(title="Test realised net profit by approval rate", xlabel="approval rate (best-ranked loans first)",
       ylabel="cumulative net profit, $ millions")
ax.legend(fontsize=8)
show_fig(fig, "profit_vs_approval_test")
print(f"[test] net profit at the champion's approval rate ({pct(pol['approval_rate'])}) by ranking:",
      {k: money(v) for k, v in at_champ.items()})

## Stress test (model-based, not realised)

The approval decisions stay fixed at the champion's validation hurdle. I then recompute **expected** profit on test under worse conditions: higher funding cost crossed with higher PDs, and separately higher funding cost crossed with a bigger loss per bad loan. The question is whether the approved book still makes money and still beats approving everyone.

In [ ]:
ST = CFG["stress"]
pd_ch = test[CHAMP_PD].to_numpy()
approved = test_masks[CHAMP]
cof_levels = [ECON["cof"] + a for a in ST["cof_add"]]


def stress_grid(param, values):
    """Expected profit of the fixed policy and of approve-all for each funding cost x stress value."""
    rows = []
    for cof in cof_levels:
        for v in values:
            ev = expected_value(test, pd_ch, cof=cof, **{param: v})
            rows.append({"cof": cof, param: v, "policy": ev[approved].sum(), "approve_all": ev.sum()})
    out = pd.DataFrame(rows)
    out["policy_beats_all"] = out["policy"] > out["approve_all"]
    return out


grid_pd, grid_bad = stress_grid("pd_mult", ST["pd_mult"]), stress_grid("bad_mult", ST["bad_mult"])


def breakeven_pd_mult(cof, mask):
    """PD multiplier at which expected profit of the approved loans reaches zero."""
    f = lambda m: expected_value(test, pd_ch, cof=cof, pd_mult=m)[mask].sum()
    if f(0.0) <= 0:
        return 0.0
    return brentq(f, 0.0, ST["breakeven_max"]) if f(ST["breakeven_max"]) < 0 else float("inf")


breakeven = pd.DataFrame({"cof": cof_levels,
                          "policy": [breakeven_pd_mult(c, approved) for c in cof_levels],
                          "approve_all": [breakeven_pd_mult(c, np.ones(len(test), bool)) for c in cof_levels]})
print("[test, model-based] expected profit ($M) with decisions fixed at the validation hurdle")
for g in (grid_pd, grid_bad):
    display(tidy(g.assign(policy=g["policy"] / 1e6, approve_all=g["approve_all"] / 1e6), 3))
print("[test, model-based] breakeven PD multiplier (inf = not reached below the search limit)")
display(tidy(breakeven, 3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), layout="constrained")
for r, (g, param, label) in enumerate([(grid_pd, "pd_mult", "PD multiplier"), (grid_bad, "bad_mult", "bad-loan loss multiplier")]):
    for c, col in enumerate(["policy", "approve_all"]):
        mat = g.pivot(index="cof", columns=param, values=col) / 1e6
        ax = axes[r, c]
        vmax = np.abs(np.r_[grid_pd[["policy", "approve_all"]].values.ravel(),
                             grid_bad[["policy", "approve_all"]].values.ravel()]).max() / 1e6
        im = ax.imshow(mat.values, cmap="RdYlGn", vmin=-vmax, vmax=vmax, origin="lower", aspect="auto")
        beats = g.pivot(index="cof", columns=param, values="policy_beats_all").values
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                note = "\nbeats all" if (col == "policy" and beats[i, j]) else ""
                ax.text(j, i, f"{mat.values[i, j]:.2f}{note}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(mat.shape[1]), [f"x{v:g}" for v in mat.columns])
        ax.set_yticks(range(mat.shape[0]), [f"{v:.0%}" for v in mat.index])
        ax.set(title=f"{CHAMP if col == 'policy' else 'approve all'}: expected profit, $M", xlabel=label,
               ylabel="cost of funds")
        ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.6, label="$ millions")
fig.savefig(FIG_DIR / "stress_heatmaps.png", dpi=120, bbox_inches="tight")
plt.show()

## Policy explorer (optional)

Sliders for the hurdle, funding cost and a PD multiplier, showing approval rate, expected loss and expected profit on test. Nothing later depends on this cell, and it is skipped if widgets are not available.

In [ ]:
try:
    import ipywidgets as widgets

    def explore(hurdle=float(HURDLES[CHAMP_MODEL]), cof=float(ECON["cof"]), pd_mult=1.0):
        """Approval rate, expected loss and expected profit for one setting."""
        ev = expected_value(test, pd_ch, cof=cof, pd_mult=pd_mult)
        ok = ev > hurdle
        p = np.clip(pd_ch * pd_mult, 0, 1)
        exp_loss = (p * test["funded_amnt"].to_numpy() * -BAD_RESULT)[ok].sum()
        print(f"approval rate {pct(ok.mean())} | expected loss {money(exp_loss)} | expected profit {money(ev[ok].sum())}")

    h0 = float(HURDLES[CHAMP_MODEL])
    lo, hi = min(float(np.quantile(test[CHAMP_EV], 0.01)), h0), max(float(np.quantile(test[CHAMP_EV], 0.99)), h0)
    widgets.interact(explore,
                     hurdle=widgets.FloatSlider(value=h0, min=lo, max=hi, step=(hi - lo) / 200),
                     cof=widgets.FloatSlider(value=float(ECON["cof"]), min=0.0, max=0.15, step=0.005),
                     pd_mult=widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.05))
except Exception as exc:
    print(f"policy explorer unavailable ({type(exc).__name__}); nothing else depends on it")

## Credit committee memo

Generated from the computed results; no number below is typed by hand.

In [ ]:
def years(df):
    """Issue-year span of a split, for plain-language labels."""
    a, b = df["issue_d"].min().year, df["issue_d"].max().year
    return f"{a}" if a == b else f"{a}-{b}"


grade_pnl = test_pnl.loc["grade policy"]
worst = grid_pd[(grid_pd["cof"] == max(cof_levels)) & (grid_pd["pd_mult"] == max(ST["pd_mult"]))].iloc[0]
be_base = float(breakeven.loc[breakeven["cof"] == ECON["cof"], "policy"].iloc[0])
psi_champ = float(score_psi.loc[score_psi["score"] == ("LightGBM PD" if CHAMP_MODEL == "lgbm" else "scorecard points"), "train->test"].iloc[0])
champ_auc = float(test_auc[CHAMP_MODEL])
summary = {
    "champion": CHAMP, "hurdle_usd": HURDLES[CHAMP_MODEL], "valid_years": years(valid), "test_years": years(test),
    "grade_cutoff": GRADE_CUT, "test_approval_rate": float(pol["approval_rate"]),
    "test_net_profit_policy": float(pol["net_profit"]), "test_net_profit_approve_all": float(base["net_profit"]),
    "test_net_profit_grade_policy": float(grade_pnl["net_profit"]), "test_roi_policy": float(pol["roi"]),
    "test_roi_approve_all": float(base["roi"]), "test_bad_rate_policy": float(pol["bad_rate"]),
    "test_bad_rate_all": float(base["bad_rate"]), "train_bad_rate": float(train["bad"].mean()),
    "test_auc_scorecard": float(test_auc["scorecard"]), "test_auc_lgbm": float(test_auc["lgbm"]),
    "test_auc_int_rate": float(test_auc["int_rate baseline"]),
    "test_ks_champion": float(metrics.loc[("test", CHAMP_MODEL), "KS"]), "test_auc_champion": champ_auc,
    "test_auc_diff_ci": [float(ci_diff[0]), float(ci_diff[1])], "auc_gap_meaningful": bool(gap_meaningful),
    "score_psi_train_test": psi_champ, "score_psi_band": psi_band(psi_champ),
    "test_mean_pd_champion": float(test[CHAMP_PD].mean()), "test_actual_bad_rate": float(test["bad"].mean()),
    "stress_worst_policy_ev": float(worst["policy"]), "stress_worst_beats_all": bool(worst["policy_beats_all"]),
    "stress_worst_cof": float(worst["cof"]), "stress_worst_pd_mult": float(worst["pd_mult"]),
    "breakeven_pd_mult_base_cof": be_base, "realization": REALIZATION, "bad_result": BAD_RESULT,
    "demo_mode": DEMO_MODE, "settings_hash": manifest["settings_hash"],
}

In [ ]:
S = summary
be_text = ("is not reached within the tested range" if np.isinf(S["breakeven_pd_mult_base_cof"])
           else f"is about {mult(S['breakeven_pd_mult_base_cof'])}")
drift_text = (f"Between training and test the bad rate moved from {pct(S['train_bad_rate'])} to {pct(S['test_actual_bad_rate'])}, "
              + ("while the score distribution itself stayed stable (PSI reads as stable), so the change came mostly from outcomes, not from a different applicant mix."
                 if S["score_psi_band"] == "stable" else
                 f"and the score distribution shifted too (PSI reads as {S['score_psi_band']}), so the applicant mix changed as well as outcomes."))
gap_pd = S["test_mean_pd_champion"] / S["test_actual_bad_rate"] - 1
calib_text = (f"The mean predicted default rate on test is {pct(S['test_mean_pd_champion'])} against an actual {pct(S['test_actual_bad_rate'])}; "
              + ("the probabilities hold up out of time" if abs(gap_pd) <= CFG["eval"]["calib_tolerance"]
                 else ("the model under-predicts because the test vintage was riskier than validation, so expected profit is optimistic"
                       if gap_pd < 0 else "the model over-predicts on test, so expected profit is conservative"))
              + ", which matters because profit is probability times dollars.")
memo = f"""
**Recommendation.** Approve a loan when the {S['champion']} model's expected profit exceeds {usd(S['hurdle_usd'])}, a hurdle chosen on the {S['valid_years']} validation vintage.
On the untouched {S['test_years']} test vintage this policy approves {pct(S['test_approval_rate'])} of loans and earns {money(S['test_net_profit_policy'])} of net profit, against {money(S['test_net_profit_approve_all'])} if every loan were approved, a change of {money(S['test_net_profit_policy'] - S['test_net_profit_approve_all'])}.
Against the incumbent grade policy (approve grades up to {S['grade_cutoff']}), which earns {money(S['test_net_profit_grade_policy'])}, the difference is {money(S['test_net_profit_policy'] - S['test_net_profit_grade_policy'])}.
Return on funded dollars moves from {pct(S['test_roi_approve_all'])} to {pct(S['test_roi_policy'])}, and the bad rate of approved loans from {pct(S['test_bad_rate_all'])} to {pct(S['test_bad_rate_policy'])}.
The model ranks risk with a test AUC of {ratio(S['test_auc_champion'])}, compared with {ratio(S['test_auc_int_rate'])} for the incumbent's own interest rate; the gap between my two models is {'meaningful' if S['auc_gap_meaningful'] else 'not statistically meaningful'}.
{calib_text}
{drift_text}
Under the harshest stress tested (cost of funds {pct(S['stress_worst_cof'])} and every PD multiplied by {mult(S['stress_worst_pd_mult'])}) the approved book's expected profit is {money(S['stress_worst_policy_ev'])}, and it {'still beats' if S['stress_worst_beats_all'] else 'no longer beats'} approving everyone.
At today's funding cost, the PD multiplier at which expected profit reaches zero {be_text}.
The main risks are that outcomes exist only for loans the original lender approved, so this policy can only tighten approvals, and that funding and operating costs are assumptions rather than measured figures.
With a lender's own data I would add reject inference, monthly repayment performance for roll rates, and MSME-specific inputs such as invoice and bank-statement data before trusting these numbers.
"""
display(Markdown(memo))

In [ ]:
write_json(REPORT_DIR / "results_summary.json", summary)
table = [("Champion policy", S["champion"]), ("Validation vintage", S["valid_years"]), ("Test vintage", S["test_years"]),
         ("Test AUC: scorecard", f"{S['test_auc_scorecard']:.3f}"), ("Test AUC: LightGBM", f"{S['test_auc_lgbm']:.3f}"),
         ("Test AUC: int_rate baseline", f"{S['test_auc_int_rate']:.3f}"), ("Test KS: champion", f"{S['test_ks_champion']:.3f}"),
         ("AUC difference (LightGBM - scorecard), 95% CI", f"{S['test_auc_diff_ci'][0]:+.3f} to {S['test_auc_diff_ci'][1]:+.3f}"),
         ("Score PSI train to test", f"{S['score_psi_train_test']:.3f} ({S['score_psi_band']})"),
         ("Mean predicted PD vs actual bad rate (test)", f"{pct(S['test_mean_pd_champion'])} vs {pct(S['test_actual_bad_rate'])}"),
         ("Approval rate (test)", pct(S["test_approval_rate"])),
         ("Net profit: champion (test)", money(S["test_net_profit_policy"])),
         ("Net profit: approve all (test)", money(S["test_net_profit_approve_all"])),
         ("Net profit: grade policy (test)", money(S["test_net_profit_grade_policy"])),
         ("ROI: champion vs approve all (test)", f"{pct(S['test_roi_policy'])} vs {pct(S['test_roi_approve_all'])}"),
         ("Bad rate approved vs all (test)", f"{pct(S['test_bad_rate_policy'])} vs {pct(S['test_bad_rate_all'])}"),
         ("Worst stress cell: policy expected profit", money(S["stress_worst_policy_ev"])),
         ("Breakeven PD multiplier at base funding cost", "not reached" if np.isinf(S["breakeven_pd_mult_base_cof"]) else f"x{S['breakeven_pd_mult_base_cof']:.2f}")]
md_table = "| Metric | Value |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in table)
note = "\n\n_DEMO_MODE run: smoke test only, not for reporting._" if DEMO_MODE else ""
(REPORT_DIR / "results_summary.md").write_text(md_table + note + "\n")

readme = Path("README.md")
start, end = "<!-- results:start -->", "<!-- results:end -->"
if readme.exists() and start in readme.read_text(encoding="utf-8"):
    text = readme.read_text(encoding="utf-8")
    text = text[: text.index(start) + len(start)] + "\n" + md_table + note + "\n" + text[text.index(end):]
    readme.write_text(text, encoding="utf-8")
    print("README results table updated from results_summary.md")
print(f"wrote {(REPORT_DIR / 'results_summary.json').relative_to(BASE).as_posix()} and results_summary.md")

## Limitations

- These are US consumer loans, not MSME credit. The method transfers; the numbers do not.
- Reject inference: outcomes exist only for loans the original lender approved, so the policy can only be tightened, and applicants they rejected are invisible.
- Time value of money is ignored in profit, and funding cost and opex are assumptions, not measured.
- Final status only, with no monthly performance data, so no roll rates or months-on-book curves.
- The EV model is simplified: one average loss-per-dollar parameter for bad loans and one prepayment factor for good loans.
- Origination features only; no behavioural or bureau-trend data.